Ten skrypt Pythona wykorzystuje LangChain do załadowania i przetworzenia dokumentu polityki HR, tworząc bazę wektorową ChromaDB oraz retriever do wyszukiwania informacji. Następnie tworzy agenta ReAct, wyposażonego w narzędzie do przeszukiwania polityki HR oraz narzędzie do konwersji walut, aby odpowiadać na złożone pytania.

# Setup

In [ ]:
!pip install -qU langchain langchain-openai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.9/433.9 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 127.7 MB/s eta 0:00:

*   `langchain`, `langchain-openai`, `langchain-community`, `pydantic`, `chromadb`: To nazwy pakietów bibliotecznych, które mają zostać zainstalowane lub zaktualizowane.

    *   `langchain`:  Główna biblioteka LangChain, framework do tworzenia aplikacji wykorzystujących modele językowe (LLM).
    *   `langchain-openai`:  Zawiera integracje specyficzne dla modeli OpenAI (np. GPT-3, GPT-4). Umożliwia łatwe korzystanie z tych modeli w aplikacjach LangChain.
    *   `langchain-community`: Zawiera integracje społecznościowe i narzędzia do pracy z różnymi modelami językowymi i źródłami danych.
    *   `pydantic`: Biblioteka służąca do walidacji danych i definiowania typów danych w Pythonie. LangChain często używa Pydantic do definiowania schematów dla wejść i wyjść modeli LLM.
    *   `chromadb`:  Baza danych wektorowa, która pozwala na efektywne przechowywanie i wyszukiwanie podobnych wektorów (reprezentacji semantycznych danych). Jest często używana w aplikacjach LangChain do implementacji mechanizmów pamięci i wyszukiwania informacji.

W skrócie, ten kod instaluje lub aktualizuje biblioteki niezbędne do tworzenia aplikacji wykorzystujących modele językowe, a konkretnie framework LangChain, wraz z integracjami dla OpenAI, społecznościowymi narzędziami, walidacją danych oraz bazą wektorową ChromaDB.

In [ ]:
import os
from langchain import hub
from langchain.agents import AgentExecutor, Tool, create_react_agent
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.document_loaders import TextLoader
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import LocalFileStore
from langchain.storage._lc_store import create_kv_docstore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.tools.retriever import create_retriever_tool
from langchain.vectorstores import Chroma
from google.colab import userdata
from pprint import pp as ppr

Ten kod importuje szereg bibliotek i modułów Pythona, które są niezbędne do budowy aplikacji wykorzystujących modele językowe (LLM) z użyciem frameworka LangChain.

*   `langchain.hub`: Moduł dostępu do "huba" LangChain, repozytorium gotowych promptów, łańcuchów (chains) i agentów.
*   `langchain.agents`: Zawiera klasy i funkcje związane z tworzeniem inteligentnych agentów, które mogą korzystać z narzędzi (tools) do wykonywania zadań.  Konkretnie importowane są:
    *   `AgentExecutor`: Klasa odpowiedzialna za uruchamianie agenta i zarządzanie jego działaniem.
    *   `Tool`: Abstrakcyjna klasa reprezentująca narzędzie, które może być używane przez agenta.
    *   `create_openai_tools_agent`, `create_react_agent`: Funkcje do tworzenia agentów wykorzystujących modele OpenAI i architekturę ReAct (Reason + Act).
    *   `tool`: Dekorator oznaczający funkcję jako narzędzie dostępne dla agenta.
*   `langchain.chains.RetrievalQA`:  Łańcuch (chain) LangChain, który łączy model językowy z mechanizmem wyszukiwania informacji w celu odpowiadania na pytania na podstawie kontekstu.
*   `langchain_openai.ChatOpenAI`, `langchain_openai.OpenAIEmbeddings`: Klasy integrujące modele OpenAI (takie jak GPT-3.5, GPT-4) i ich embeddingi (reprezentacje wektorowe tekstu).
*   `langchain.document_loaders.TextLoader`:  Klasa do wczytywania dokumentów tekstowych z plików.
*   `langchain.embeddings.openai.OpenAIEmbeddings`: Klasa do generowania embeddingów za pomocą modeli OpenAI. (Duplikacja importu, prawdopodobnie niepotrzebna).
*   `langchain.prompts.ChatPromptTemplate`, `langchain.prompts.MessagesPlaceholder`:  Klasy do tworzenia szablonów promptów dla modeli językowych w formacie rozmowy (chat).
*   `langchain.retrievers.ParentDocumentRetriever`: Klasa służąca do pobierania dokumentów z uwzględnieniem ich hierarchicznej struktury.
*   `langchain.storage.LocalFileStore`, `langchain.storage._lc_store.create_kv_docstore`:  Klasy i funkcje związane z przechowywaniem danych w lokalnym systemie plików.
*   `langchain.text_splitter.RecursiveCharacterTextSplitter`: Klasa do dzielenia długich tekstów na mniejsze fragmenty, aby dopasować je do ograniczeń kontekstu modelu językowego.
*   `langchain.tools.retriever.create_retriever_tool`: Funkcja tworząca narzędzie (tool) LangChain, które umożliwia agentowi wyszukiwanie informacji za pomocą retrievera.
*   `langchain.vectorstores.Chroma`: Klasa do pracy z bazą danych wektorowej ChromaDB, która przechowuje embeddingi dokumentów i umożliwia szybkie wyszukiwanie podobnych fragmentów tekstu.
*   `google.colab.userdata`: Moduł dostępu do danych użytkownika w środowisku Google Colab.
*   `pprint.pp as ppr`:  Moduł "pretty print" (ładne drukowanie) służący do wyświetlania struktur danych w czytelny sposób. Alias `ppr` jest używany dla wygody.

Ten kod przygotowuje środowisko programistyczne poprzez zaimportowanie wszystkich niezbędnych bibliotek i modułów do budowy aplikacji wykorzystujących LLM z frameworkiem LangChain, obejmujących ładowanie danych, generowanie embeddingów, tworzenie agentów, łańcuchów i narzędzi oraz przechowywanie danych.


In [ ]:
class CFG:
    model1 = "gpt-4o-mini"
    model2 = "text-embedding-3-large"
    temperature = 0.1
    chunksize = 1024

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")

In [ ]:
llm = ChatOpenAI(model=CFG.model1)

embeddings = OpenAIEmbeddings(model=CFG.model2)

<ipython-input-5-2be88f6e64f2>:3: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(model = CFG.model2)


# Dane

In [ ]:
loader = TextLoader("./content/globalcorp_hr_policy.txt")
documents = loader.load()

Ten kod ładuje zawartość pliku tekstowego i przekształca ją w listę dokumentów, które mogą być przetwarzane przez framework LangChain.

In [ ]:
persist_directory = "local_vectorstore"
local_store = "local_docstore"
collection_name = "hrpolicy"
PROJECT_ROOT = "/content/"

Ten kod definiuje zmienne przechowujące informacje dotyczące lokalizacji i nazewnictwa elementów związanych z przechowywaniem danych wektorowych.

*   `persist_directory = "local_vectorstore"`: Definiuje zmienną `persist_directory` i przypisuje jej wartość `"local_vectorstore"`. Ta zmienna określa katalog, w którym będą przechowywane pliki bazy danych wektorowej ChromaDB.
*   `local_store = "local_docstore"`: Definiuje zmienną `local_store` i przypisuje jej wartość `"local_docstore"`.  Ta zmienna prawdopodobnie odnosi się do nazwy lokalnego magazynu dokumentów, który będzie używany do przechowywania metadanych lub innych informacji związanych z dokumentami.
*   `collection_name = "hrpolicy"`: Definiuje zmienną `collection_name` i przypisuje jej wartość `"hrpolicy"`. Ta zmienna określa nazwę kolekcji w bazie danych wektorowej ChromaDB, która będzie przechowywać embeddingi i metadane dokumentów dotyczących polityki personalnej.
*   `PROJECT_ROOT = "/content/"`: Definiuje zmienną `PROJECT_ROOT` i przypisuje jej wartość `"/content/"`. Ta zmienna określa katalog główny projektu w środowisku Google Colab. Jest to często używane jako punkt odniesienia do ścieżek plików.

Ten fragment kodu definiuje zmienne konfiguracyjne dotyczące lokalizacji bazy danych wektorowej ChromaDB, magazynu dokumentów i nazwy kolekcji, a także katalogu głównego projektu w środowisku Google Colab. Ułatwia to zarządzanie plikami i danymi w aplikacji.

# Retriever

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

Ten kod definiuje dwa obiekty `RecursiveCharacterTextSplitter`, które będą używane do dzielenia tekstu na mniejsze fragmenty (chunki). Dzielenie tekstu jest konieczne, ponieważ modele językowe mają ograniczenia dotyczące długości przetwarzanego tekstu.


In [ ]:
vectorstore = Chroma(
    persist_directory=os.path.join(PROJECT_ROOT, "data", persist_directory),
    collection_name=collection_name,
    embedding_function=embeddings,
)

<ipython-input-9-b89b5b0f54e8>:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


Ten kod inicjalizuje bazę danych wektorową ChromaDB i konfiguruje ją do przechowywania embeddingów dokumentów.

In [ ]:
local_store = LocalFileStore(os.path.join(PROJECT_ROOT, "data", local_store))
store = create_kv_docstore(local_store)
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

Ten kod konfiguruje mechanizm wyszukiwania i pobierania dokumentów z wykorzystaniem bazy danych wektorowej ChromaDB oraz lokalnego magazynu dokumentów.

*   `local_store = LocalFileStore(os.path.join(PROJECT_ROOT, "data", local_store))`:
    *   `LocalFileStore`: Klasa z biblioteki `langchain.storage`, która służy do przechowywania danych w lokalnym systemie plików.
    *   `os.path.join(...)`: Funkcja łącząca ścieżki, tworząca pełną ścieżkę do katalogu magazynu: `/content/data/local_docstore`.
    *   Tworzy instancję `LocalFileStore`, która będzie przechowywać metadane i inne informacje związane z dokumentami.

*   `store = create_kv_docstore(local_store)`:
    *   `create_kv_docstore`: Funkcja tworząca magazyn klucz-wartość (key-value) do przechowywania dokumentów w `LocalFileStore`.
    *   Tworzy obiekt `store`, który będzie używany do przechowywania i pobierania dokumentów na podstawie ich identyfikatorów.

*   `retriever = ParentDocumentRetriever(...)`:
    *   `ParentDocumentRetriever`: Klasa z biblioteki `langchain.retrievers`, która służy do pobierania dokumentów w sposób hierarchiczny, wykorzystując zarówno bazę danych wektorową, jak i lokalny magazyn dokumentów.
    *   `vectorstore=vectorstore`: Określa bazę danych wektorową ChromaDB, z której będą pobierane embeddingi i informacje o podobieństwie.
    *   `docstore=store`: Określa lokalny magazyn dokumentów, który będzie używany do przechowywania metadanych i innych informacji związanych z dokumentami.
    *   `child_splitter=child_splitter`: Określa splitter tekstu, który będzie używany do dzielenia większych fragmentów tekstu na mniejsze chunki (ustawiony wcześniej na 200 znaków).
    *   `parent_splitter=parent_splitter`: Określa splitter tekstu, który będzie używany do dzielenia dokumentów na większe chunki (ustawiony wcześniej na 1000 znaków).



In [ ]:
vectorstore.persist()
retriever.add_documents(documents, ids=None)

<ipython-input-11-431b951c94f6>:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


Ten kod zapisuje bazę danych wektorową na dysku i dodaje do niej dokumenty, które zostały wcześniej załadowane.

*   `vectorstore.persist()`:
    *   `vectorstore`: Obiekt reprezentujący bazę danych wektorowej ChromaDB (utworzony wcześniej).
    *   `.persist()`: Metoda klasy `Chroma`, która zapisuje zawartość bazy danych na dysku w katalogu określonym podczas inicjalizacji (`persist_directory`). Dzięki temu dane zostaną zachowane między sesjami i będą dostępne po ponownym uruchomieniu programu.

*   `retriever.add_documents(documents, ids = None)`:
    *   `retriever`: Obiekt `ParentDocumentRetriever`, który jest odpowiedzialny za pobieranie dokumentów z bazy danych wektorowej i lokalnego magazynu dokumentów.
    *   `.add_documents(documents, ids = None)`: Metoda klasy `ParentDocumentRetriever`, która dodaje nowe dokumenty do bazy danych wektorowej i lokalnego magazynu dokumentów.
        *   `documents`: Lista obiektów typu `Document`, które mają zostać dodane (załadowane wcześniej z pliku `/content/globalcorp_hr_policy.txt`).
        *   `ids = None`: Określa identyfikatory dla każdego dokumentu. Jeśli wartość jest ustawiona na `None`, zostaną wygenerowane automatyczne identyfikatory.


# Functions

In [ ]:
def value_to_float(x):
    if int(x):
        return int(x)
    if type(x) is float or type(x) is int:
        return x
    x = x.upper()
    if "MILLION" in x:
        if len(x) > 1:
            return float(x.replace("MILLION", "")) * 1000000
        return 1000000.0
    if "BILLION" in x:
        return float(x.replace("BILLION", "")) * 1000000000

    return 0.0


Ten kod definiuje funkcję o nazwie `value_to_float`, która próbuje przekonwertować wartość wejściową na liczbę zmiennoprzecinkową (float). Funkcja obsługuje różne formaty wartości, w tym liczby całkowite, liczby zmiennoprzecinkowe oraz wartości tekstowe zawierające słowa "MILLION" lub "BILLION".

In [ ]:
def convert_currency_to_usd(amount: str) -> int:
    "Converts currency into USD"

    if "¥" in amount:
        exclude_symbol = amount.replace("¥", "")
        amount_in_numbers = value_to_float(exclude_symbol)
        return amount_in_numbers / 147.72
    if "$" in amount:
        exclude_symbol = amount.replace("$", "")
        return value_to_float(exclude_symbol)
    if "JPY" in amount:
        exclude_symbol = amount.replace("JPY", "")
        return int(exclude_symbol) / 147.72
    if "USD" in amount:
        return amount


# QnA system

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

Ten kod tworzy łańcuch (chain) typu RetrievalQA, który służy do odpowiadania na pytania na podstawie kontekstu pobranego z bazy danych wektorowej.

## Plusy

In [ ]:
response = qa({"query": "What is the allocated budget for communication initiatives?"})
ppr(response)

<ipython-input-15-f7edc09cb693>:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa({"query": "What is the allocated budget for communication initiatives?"})


{'query': 'What is the allocated budget for communication initiatives?',
 'result': 'The allocated budget for communication initiatives is $500,000 in '
           'the U.S., €250,000 in Germany, and ¥25 million in Japan.',
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='**Health and Safety:**\nWorkplace safety is a shared responsibility. Emergency procedures are clearly posted throughout our office buildings, and an annual budget of $10,000, €5,000, and ¥1 million is allocated for safety drills and equipment maintenance in the U.S., Germany, and Japan, respectively.\n\n**Communication:**\nImportant updates are conveyed through company-wide emails and team meetings. An annual budget of $500,000, €250,000, and ¥25 million is allocated for communication initiatives, including employee engagement events in the U.S., Germany, and Japan, respectively.\n\nThis policy undergoes an annual review to ensure relevance and compliance. Welcome 

In [ ]:
qa({"query": "How many maternity leaves are offered?"})

{'query': 'How many maternity leaves are offered?',
 'result': "I don't know.",
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='2.4\t<Company> may, on occasions, decide to restrict advertisement to internal candidates only. Furthermore junior posts will always be advertised internally in the first instance, to provide continuous development of existing members of staff unless the Management team agrees that this is not appropriate due to the specialist skills required for the post involved. \n\n2.5\tVacancies, which are restricted to internal candidates only will be clearly indicated on the advertisement. All internal candidates will be selected for interview on the same criteria as external candidates. \n\n2.6\tEmployees on maternity leave will receive all advertisements for posts advertised in the Company during their period of maternity leave.\n3.\tApplication Form\n\n3.1\tCandidates for all posts will, except on some occasions 

In [ ]:
qa({"query": "What is the probationary period?"})

{'query': 'What is the probationary period?',
 'result': "The probationary period is a six-calendar-month period for all appointments into the Company. After three months, a review meeting will take place between the post holder and their line manager to discuss progress. At the end of the probationary period, if there is a satisfactory report from the appropriate head of section or line manager, employees will be notified in writing that they have successfully completed their probationary period. Additionally, the probationary period can be extended by a further three months if the individual's line manager considers it appropriate.",
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='•\ttwo satisfactory references;\n•\tconfirmation of the right to work in this country (if appropriate)\n•\tCriminal Records Disclosure (if appropriate).\n6.\tProbationary Period\n\t\n6.1\tAll appointments into the Company will be made subject to a proba

In [ ]:
qa({"query": "Is the work hours in Germany different than United States?"})

{'query': 'Is the work hours in Germany different than United States?',
 'result': 'Yes, the work hours in Germany are different than in the United States. In Germany, the standard workweek is 38 hours (Monday to Friday, 8 AM to 5 PM), whereas in the United States, employees adhere to a standard 40-hour workweek (Monday to Friday, 9 AM to 5 PM).',
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='**GlobalCorp Human Resources Policy**\n\nWelcome to GlobalCorp, where our Human Resources Policy is designed to provide a comprehensive framework for employees across our offices in the United States, Germany, and Japan. We operate under an at-will employment relationship, and any contractual agreements should be documented in writing.\n\nAt the core of our culture is a commitment to professionalism and ethical conduct. Clear and respectful communication is highly valued, and a business casual dress code is encouraged.\n\n**Work Hours:**\nEm

In [ ]:
qa({"query": "What is the probationary period for new employees in USA?"})

{'query': 'What is the probationary period for new employees in USA?',
 'result': "I don't know.",
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='•\ttwo satisfactory references;\n•\tconfirmation of the right to work in this country (if appropriate)\n•\tCriminal Records Disclosure (if appropriate).\n6.\tProbationary Period\n\t\n6.1\tAll appointments into the Company will be made subject to a probationary period of six calendar months. After three months a review meeting will take place between the post holder and their line manager to discuss progress. At the end of the probationary period, and subject to a satisfactory report by the appropriate head of section or line manager, employees will be notified in writing that they have successfully completed their probationary period. The probationary period can be extended by a further 3 months should the individuals line manager consider this appropriate.\n7.\tRecruitment Monitoring')]

In [ ]:
qa(
    {
        "query": "What is the difference in the number of work hours in Germany vs. United States?"
    }
)

{'query': 'What is the difference in the number of work hours in Germany vs. United States?',
 'result': 'The standard workweek in Germany is 38 hours, while in the United States it is 40 hours. Therefore, the difference in the number of work hours is 2 hours per week, with the U.S. having a longer workweek.',
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='**GlobalCorp Human Resources Policy**\n\nWelcome to GlobalCorp, where our Human Resources Policy is designed to provide a comprehensive framework for employees across our offices in the United States, Germany, and Japan. We operate under an at-will employment relationship, and any contractual agreements should be documented in writing.\n\nAt the core of our culture is a commitment to professionalism and ethical conduct. Clear and respectful communication is highly valued, and a business casual dress code is encouraged.\n\n**Work Hours:**\nEmployees in the United States adhere to

In [ ]:
qa({"query": "Can I reimburse travel expenses?"})

{'query': 'Can I reimburse travel expenses?',
 'result': 'Yes, candidates attending an interview will be reimbursed any reasonable expenses incurred, such as travelling expenses which would normally be at the appropriate second-class rail fare. However, a candidate who withdraws or refuses an offer of appointment for reasons considered inadequate will not be reimbursed.',
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='4.6\t\t\tIt should be remembered that an interview is a two way process, and candidates will be given every opportunity to view the offices where they will work and ask questions about the Company, to ensure that they have a full understanding of the post for which they are applying and the way the Company operates.\n\n4.7\t\t\tIn addition to interviews, a range of other selection techniques may be used. In such circumstances reasonable notice and relevant information will be given to ensure that candidates have suff

In [ ]:
qa(
    {
        "query": "I started with the company on 1 December 2023.\
    Is my probationary period over if the date today is 26 Jan 2024?"
    }
)

{'query': 'I started with the company on 1 December 2023.    Is my probationary period over if the date today is 26 Jan 2024?',
 'result': 'No, your probationary period is not over yet. Since you started on 1 December 2023, your six-month probationary period would end on 1 June 2024.',
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='•\ttwo satisfactory references;\n•\tconfirmation of the right to work in this country (if appropriate)\n•\tCriminal Records Disclosure (if appropriate).\n6.\tProbationary Period\n\t\n6.1\tAll appointments into the Company will be made subject to a probationary period of six calendar months. After three months a review meeting will take place between the post holder and their line manager to discuss progress. At the end of the probationary period, and subject to a satisfactory report by the appropriate head of section or line manager, employees will be notified in writing that they have successfully comp

## Minusy

In [ ]:
# złe przeliczenie
qa(
    {
        "query": "What is the percentage difference in the annual budget for Japan and US?"
    }
)

{'query': 'What is the percentage difference in the annual budget for Japan and US?',
 'result': "To find the percentage difference in the annual budget for Japan and the U.S., we need to compare the two budgets for employee benefits and leave-related expenses.\n\nFor Japan:\n- Employee benefits budget: ¥50 million\n- Leave-related expenses budget: ¥100 million\n- Total budget for Japan: ¥150 million\n\nFor the U.S.:\n- Employee benefits budget: $1 million\n- Leave-related expenses budget: $500,000\n- Total budget for the U.S.: $1.5 million\n\nNow, we will convert both totals into the same currency for easy comparison. Since the budgets are in different currencies (Yen and USD), we would typically need the current exchange rate to accurately convert them. However, since the information does not provide an exchange rate, it is impossible to calculate an accurate percentage difference.\n\nTherefore, I don't know the percentage difference without additional information on currency convers

In [ ]:
qa(
    {
        "query": "What is the percentage difference in the annual budget for Japan and US if 1 USD = 147.72 JPY?"
    }
)

{'query': 'What is the percentage difference in the annual budget for Japan and US if 1 USD = 147.72 JPY?',
 'result': 'To find the percentage difference in the annual budget for Japan and the US, we first need to convert the budgets into the same currency. \n\nThe annual budget for employee benefits in the US is $1 million, which is equivalent to:\n\n1,000,000 USD * 147.72 JPY/USD = 147,720,000 JPY\n\nThe annual budget for employee benefits in Japan is ¥50 million.\n\nNow, we can find the percentage difference between the two budgets. The formula for percentage difference is:\n\nPercentage Difference = (|Value1 - Value2| / Average of Value1 and Value2) * 100\n\nHere, Value1 = 147,720,000 JPY (US budget), and Value2 = 50,000,000 JPY (Japan budget).\n\nCalculating the average:\n\nAverage = (147,720,000 + 50,000,000) / 2 = 98,860,000 JPY\n\nCalculating the absolute difference:\n\n|Value1 - Value2| = |147,720,000 - 50,000,000| = 97,720,000 JPY\n\nNow substituting into the percentage diffe

In [ ]:
qa({"query": "Which country has the highest budget?"})

{'query': 'Which country has the highest budget?',
 'result': 'The United States has the highest budget allocations in several categories, including $1 million for cybersecurity training and awareness programs, $1 million for employee benefits, and $500,000 for leave-related expenses.',
 'source_documents': [Document(metadata={'source': '/content/globalcorp_hr_policy.txt'}, page_content='**Grievance and Disciplinary Procedures:**\nOur grievance and disciplinary procedures are outlined on the company intranet. Termination conditions may include gross misconduct or repeated policy violations. In such cases, a disciplinary process will be followed, including a three-strike system, before termination. Employees leaving GlobalCorp should follow the exit process detailed in the employee handbook.\n\n**Confidentiality and Data Security:**\nEmployees are expected to maintain confidentiality, and strict data security measures are in place. An annual budget of $1 million is allocated for cyberse

# Agent

In [ ]:
tool_search = create_retriever_tool(
    retriever,
    "search_hr_policy",
    "Searches and returns excerpts from the HR policy.",
)

Ten kod tworzy narzędzie (tool) dla agenta LangChain, które umożliwia wyszukiwanie informacji w bazie danych wektorowej zawierającej politykę personalną.

*   `tool_search = create_retriever_tool(...)`: Tworzy instancję narzędzia za pomocą funkcji `create_retriever_tool`.
*   `retriever`: Określa retriever, który będzie używany do wyszukiwania dokumentów. W tym przypadku jest to obiekt `retriever`, który został utworzony wcześniej z klasy `ParentDocumentRetriever`.
*   `"search_hr_policy"`:  Określa nazwę narzędzia. Ta nazwa będzie używana przez agenta do identyfikacji i wywoływania tego narzędzia.
*   `"Searches and returns excerpts from the HR policy."`: Określa opis narzędzia. Ten opis jest wyświetlany agentowi, aby pomóc mu zrozumieć, jak działa to narzędzie i kiedy go używać.


In [ ]:
currency_conversion = Tool(
    name="Currency_conversion",
    func=convert_currency_to_usd,
    description="useful for converting currency into USD. Input should be an amount.",
)

Ten kod tworzy narzędzie (tool) dla agenta LangChain, które umożliwia konwersję walut na dolary amerykańskie (USD).

*   `currency_conversion = Tool(...)`: Tworzy instancję klasy `Tool`.
*   `name="Currency_conversion"`: Określa nazwę narzędzia. Ta nazwa będzie używana przez agenta do identyfikacji i wywoływania tego narzędzia.
*   `func=convert_currency_to_usd`:  Określa funkcję, która ma być wywołana, gdy agent użyje tego narzędzia. W tym przypadku jest to funkcja `convert_currency_to_usd`, która została wcześniej zdefiniowana i konwertuje kwoty w różnych walutach na dolary amerykańskie.
*   `description="useful for converting currency into USD. Input should be an amount."`: Określa opis narzędzia. Ten opis jest wyświetlany agentowi, aby pomóc mu zrozumieć, jak działa to narzędzie i kiedy go używać. Opis informuje, że narzędzie służy do konwersji walut na dolary amerykańskie i oczekuje jako wejścia kwoty.


In [ ]:
prompt_react = hub.pull("hwchase17/react")

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:280: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Ten kod pobiera szablon promptu z "huba" LangChain i przypisuje go do zmiennej `prompt_react`.

*   `prompt_react = hub.pull("hwchase17/react")`:
    *   `hub.pull(...)`: Funkcja z biblioteki `langchain.hub`, która pobiera zasób (np. prompt, łańcuch, agent) z huba LangChain. Hub to repozytorium gotowych komponentów, które można wykorzystać w aplikacjach LangChain.
    *   `"hwchase17/react"`:  Identyfikator zasobu, który ma zostać pobrany. W tym przypadku jest to prompt o nazwie "react", udostępniony przez użytkownika "hwchase17". Ten prompt jest przeznaczony do użycia z agentami działającymi w architekturze ReAct (Reason + Act).
    *   `prompt_react = ...`: Przypisuje pobrany szablon promptu do zmiennej `prompt_react`.


In [ ]:
prompt_react

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:
# jedno narzędzie
tools = [tool_search]

react_agent = create_react_agent(llm, tools=tools, prompt=prompt_react)

Ten kod tworzy agenta LangChain wykorzystującego architekturę ReAct (Reason + Act) i skonfigurowanego do korzystania z narzędzia wyszukiwania polityki personalnej.

*   `tools = [tool_search]`: Tworzy listę narzędzi, które będą dostępne dla agenta. W tym przypadku lista zawiera tylko jedno narzędzie: `tool_search`, które zostało utworzone wcześniej i umożliwia wyszukiwanie fragmentów tekstu z polityki personalnej.
*   `react_agent = create_react_agent(llm, tools=tools, prompt=prompt_react)`: Tworzy instancję agenta ReAct za pomocą funkcji `create_react_agent`.
    *   `llm`: Określa model językowy, który będzie używany przez agenta do generowania rozumowań i działań. W tym przypadku jest to obiekt `llm`, który został utworzony wcześniej z klasy `ChatOpenAI`.
    *   `tools=tools`: Określa listę narzędzi, które są dostępne dla agenta.
    *   `prompt=prompt_react`: Określa szablon promptu, który będzie używany do sterowania działaniem agenta. W tym przypadku jest to szablon `prompt_react`, który został pobrany wcześniej z huba LangChain.


In [ ]:
react_agent_executor = AgentExecutor(
    agent=react_agent, tools=tools, verbose=True, handle_parsing_errors=True
)

Ten kod tworzy wykonawca agenta (Agent Executor), który zarządza działaniem agenta ReAct i umożliwia interakcję z nim.

*   `react_agent_executor = AgentExecutor(...)`: Tworzy instancję klasy `AgentExecutor`.
*   `agent=react_agent`: Określa agenta, którego ma zarządzać wykonawca. W tym przypadku jest to obiekt `react_agent`, który został utworzony wcześniej z klasy `create_react_agent`.
*   `tools=tools`: Określa listę narzędzi, które są dostępne dla agenta. Jest to ta sama lista narzędzi, która została przekazana do funkcji `create_react_agent`.
*   `verbose=True`: Ustawia poziom szczegółowości (verbosity) na True. Oznacza to, że wykonawca będzie wyświetlać informacje o każdym kroku działania agenta, takie jak rozumowanie, wybrane narzędzie i wynik jego wykonania. Jest to przydatne do debugowania i monitorowania działania agenta.
*   `handle_parsing_errors=True`: Włącza obsługę błędów parsowania. Jeśli agent napotka błąd podczas próby sparsowania (zinterpretowania) wyniku działania narzędzia, wykonawca spróbuje go obsłużyć i kontynuować działanie.


In [ ]:
query = "As per the HR policy, Which country has the highest budget?"
react_agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
To find the answer, I need to search the HR policy for any information regarding budget allocations by country. 

Action: search_hr_policy  
Action Input: "highest budget by country"  **Grievance and Disciplinary Procedures:**
Our grievance and disciplinary procedures are outlined on the company intranet. Termination conditions may include gross misconduct or repeated policy violations. In such cases, a disciplinary process will be followed, including a three-strike system, before termination. Employees leaving GlobalCorp should follow the exit process detailed in the employee handbook.

**Confidentiality and Data Security:**
Employees are expected to maintain confidentiality, and strict data security measures are in place. An annual budget of $1 million is allocated for cybersecurity training and awareness programs in the U.S., while budgets for Germany and Japan are €500,000 and ¥50 million, respectively.

**Health and Safety:**
Workplace safet

{'input': 'As per the HR policy, Which country has the highest budget?',
 'output': 'The country with the highest budget is the United States, with an allocation of $1 million for cybersecurity training and awareness programs.'}

In [ ]:
query = "Which of the two countries has the highest budget - Japan or Unites States?"
react_agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
I need to find recent information about the budget of Japan and the United States to compare the two. Since this information is not related to HR policies, I cannot utilize the search tool provided. Instead, I will rely on my existing knowledge. 

The United States typically has a higher budget compared to Japan due to its larger economy and higher military expenditures. The U.S. budget often exceeds $4 trillion, while Japan's is generally lower, around $1 trillion to $1.5 trillion.

Thought: Given my knowledge, the United States clearly has the higher budget.
Final Answer: The United States has the highest budget compared to Japan.

> Finished chain.


{'input': 'Which of the two countries has the highest budget - Japan or Unites States?',
 'output': 'The United States has the highest budget compared to Japan.'}

In [ ]:
query = "How much is the budget for Japan different than United States?"
react_agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
To answer the question regarding the budget differences between Japan and the United States, I will need to gather relevant information or documents that reflect the HR policy or budgetary details of these two countries. Since it appears that this specific query may not fall under HR policy specifically, I will check the available HR policy documentation to see if it includes any budgetary comparisons or relevant context.

Action: search_hr_policy
Action Input: 'budget comparison between Japan and United States'
**Grievance and Disciplinary Procedures:**
Our grievance and disciplinary procedures are outlined on the company intranet. Termination conditions may include gross misconduct or repeated policy violations. In such cases, a disciplinary process will be followed, including a three-strike system, before termination. Employees leaving GlobalCorp should follow the exit process detailed in the employee handbook.

**Confidentiality and Data Secu

{'input': 'How much is the budget for Japan different than United States?',
 'output': 'The budget for cybersecurity training and employee benefits in the United States is $500,000 higher than that of Japan, while Japan has $500,000 more allocated for leave-related expenses compared to the United States.'}

In [ ]:
# dwa
tools = [tool_search, currency_conversion]

react_agent = create_react_agent(llm, tools=tools, prompt=prompt_react)
react_agent_executor = AgentExecutor(
    agent=react_agent, tools=tools, verbose=True, handle_parsing_errors=True
)

Ten kod aktualizuje konfigurację agenta LangChain, dodając do niego narzędzie konwersji walut.

*   `tools = [tool_search, currency_conversion]`: Tworzy nową listę narzędzi, która zawiera zarówno narzędzie wyszukiwania polityki personalnej (`tool_search`), jak i narzędzie konwersji walut (`currency_conversion`).
*   `react_agent = create_react_agent(llm, tools=tools, prompt=prompt_react)`: Ponownie tworzy agenta ReAct, ale tym razem z nową listą narzędzi. Dzięki temu agent będzie mógł korzystać zarówno z wyszukiwania polityki personalnej, jak i konwersji walut.
*   `react_agent_executor = AgentExecutor(...)`: Ponownie tworzy wykonawcę agenta, używając nowego agenta ReAct i nowej listy narzędzi. Pozostałe parametry (verbose=True, handle_parsing_errors=True) pozostają bez zmian.


In [ ]:
query = "Is the budget for Japan different than United States?"
react_agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
To determine if the budget for Japan is different from that of the United States, I need to check the HR policy regarding budget provisions or allocations for different countries, specifically looking at Japan and the United States. 

Action: search_hr_policy  
Action Input: 'budget Japan United States'  **Grievance and Disciplinary Procedures:**
Our grievance and disciplinary procedures are outlined on the company intranet. Termination conditions may include gross misconduct or repeated policy violations. In such cases, a disciplinary process will be followed, including a three-strike system, before termination. Employees leaving GlobalCorp should follow the exit process detailed in the employee handbook.

**Confidentiality and Data Security:**
Employees are expected to maintain confidentiality, and strict data security measures are in place. An annual budget of $1 million is allocated for cybersecurity training and awareness programs in the U.S

{'input': 'Is the budget for Japan different than United States?',
 'output': 'Yes, the budget for Japan is different than that of the United States. For employee benefits, Japan has an annual budget of ¥50 million, while the United States has a budget of $1 million. Additionally, the budget allocated for leave-related expenses in Japan is ¥100 million, compared to the $500,000 budget in the United States.'}